In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import os
import time
import numpy as np
import cv2
import seaborn as sns
import keras
import keras_tuner as kt

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.layers import Rescaling
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras import models, layers, Sequential
from keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.models import load_model
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
TRAIN_DATA_PATH = "../affectnet_dataset/Train"
TEST_DATA_PATH = "../affectnet_dataset/Test"
ORIGINAL_DATA_PATH = "../affectnet_dataset"

# CLAHE_DATASET_PATH = "../clahe_dataset"
# MASK_DATASET_PATH = "../mask_dataset"
EPOCHS = 100
RANDOM_SEED = 40
BATCH_SIZE = 32
IMG_SIZE = (96,96)

SAVED_MODEL = "cnn_weighted_model.h5"
# CLASSES = [d for d in os.listdir(TRAIN_DATA_PATH) if d != '.DS_Store']

### CLAHE PREPROCESSING

In [3]:
# def apply_clahe(img):
#     img = img.astype(np.uint8)

#     lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
#     l, a, b = cv2.split(lab)

#     clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
#     l = clahe.apply(l)

#     lab = cv2.merge((l, a, b))
#     img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

#     return img


# def process_and_save_clahe(input_root, output_root, img_size=(96,96)):
#     for split in ["Train", "Test"]:
#         split_path = os.path.join(input_root, split)

#         for class_name in os.listdir(split_path):
#             class_path = os.path.join(split_path, class_name)

#             if not os.path.isdir(class_path):
#                 continue

#             save_class_path = os.path.join(output_root, split, class_name)
#             os.makedirs(save_class_path, exist_ok=True)

#             for img_name in os.listdir(class_path):
#                 if img_name.startswith("."):
#                     continue

#                 img_path = os.path.join(class_path, img_name)

#                 img = cv2.imread(img_path)
#                 if img is None:
#                     continue

#                 img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#                 img = cv2.resize(img, img_size)

#                 img = apply_clahe(img)

#                 img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

#                 save_path = os.path.join(save_class_path, img_name)
#                 cv2.imwrite(save_path, img)

#     print("CLAHE train and test dataset saved!")

### UNSHARP MASKING PREPROCESSING

In [4]:
# def unsharp_mask(img, sigma=1.0, amount=1.2):
#     # calculates the optimal kernel size (w,h) from the provided sigma.
#     blurred = cv2.GaussianBlur(img, (0,0), sigma)
#     # 1 + amount (alpha): The weight given to the original image
#     # -amount (beta): The weight given to the blurred image
#     # gamma at 0 (no brightness adjustment)
#     sharp = cv2.addWeighted(img, 1+amount, blurred, -amount, 0)
#     return sharp

# def process_and_save_mask(input_root, output_root, img_size=(96,96)):
#     for split in ["Train"]:
#         split_path = os.path.join(input_root, split)

#         for class_name in os.listdir(split_path):
#             class_path = os.path.join(split_path, class_name)

#             if not os.path.isdir(class_path):
#                 continue

#             save_class_path = os.path.join(output_root, split, class_name)
#             os.makedirs(save_class_path, exist_ok=True)

#             for img_name in os.listdir(class_path):
#                 if img_name.startswith("."):
#                     continue

#                 img_path = os.path.join(class_path, img_name)

#                 img = cv2.imread(img_path)
#                 if img is None:
#                     continue

#                 img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#                 img = cv2.resize(img, img_size)

#                 img = unsharp_mask(img)

#                 img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

#                 save_path = os.path.join(save_class_path, img_name)
#                 cv2.imwrite(save_path, img)

#     print("Mask train dataset saved: ", output_root)

In [5]:
# process_and_save_clahe(ORIGINAL_DATA_PATH, CLAHE_DATASET_PATH, img_size=IMG_SIZE)
# process_and_save_mask(ORIGINAL_DATA_PATH, MASK_DATASET_PATH, img_size=IMG_SIZE)

In [6]:
# TRAIN_CLAHE_DATA_PATH = "../clahe_dataset/Train"
# TEST_CLAHE_DATA_PATH = "../clahe_dataset/Test"
# TRAIN_MASK_DATA_PATH = "../mask_dataset/Train"
# TEST_MASK_DATA_PATH = "../mask_dataset/Test"

In [7]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(
    rescale=1./255,
)

In [8]:
train_dataset = train_datagen.flow_from_directory(
    TRAIN_DATA_PATH,
    subset="training",
    seed = RANDOM_SEED,
    target_size=IMG_SIZE,
    color_mode="rgb",
    class_mode = "sparse"
)

val_dataset = train_datagen.flow_from_directory(
    TRAIN_DATA_PATH,
    seed = RANDOM_SEED,
    target_size=IMG_SIZE,
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    class_mode = "sparse",
    shuffle=False,
    subset="validation"
)
test_dataset = test_datagen.flow_from_directory(
    TEST_DATA_PATH,
    target_size=IMG_SIZE,
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    shuffle=False,
    class_mode = "sparse"
)

Found 12890 images belonging to 8 classes.
Found 3218 images belonging to 8 classes.
Found 14518 images belonging to 8 classes.


In [9]:
print(train_dataset.classes)
print(train_dataset.class_indices.values())
train_labels=train_dataset.classes
print(np.unique(train_labels))

[0 0 0 ... 7 7 7]
dict_values([0, 1, 2, 3, 4, 5, 6, 7])
[0 1 2 3 4 5 6 7]


In [10]:
'''
Calculate class weights for imbalanced dataset
Create label list manually from folder counts
Class weights змушують loss сильніше штрафувати помилки на rare classes.
'''

train_labels = train_dataset.classes

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = dict(enumerate(class_weights_arr))
print(class_weights)

{0: np.float64(1.3427083333333334), 1: np.float64(1.291065705128205), 2: np.float64(1.63744918699187), 3: np.float64(1.3316115702479339), 4: np.float64(0.8607104700854701), 5: np.float64(0.7300634345265066), 6: np.float64(0.6515365952284674), 7: np.float64(0.9500294811320755)}


In [11]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

### Keras_tuner: Hyperparameter Tuning

In [12]:
# def build_model(hp):
#     model = tf.keras.Sequential([
#         layers.Input(shape=(*IMG_SIZE, 3)),
#         data_augmentation,
#         # Block 1
#         layers.Conv2D(hp.Choice('filters1', [32,64]), (3,3), padding='same'),
#         layers.BatchNormalization(),
#         layers.Activation('relu'),
#         layers.Conv2D(hp.Choice('filters2', [32,64]), (3,3), padding='same'),
#         layers.BatchNormalization(),
#         layers.Activation('relu'),
#         layers.MaxPooling2D(),
#         layers.Dropout(hp.Float('dropout1', 0.1, 0.5, step=0.1)),

#         # Block 2
#         layers.Conv2D(hp.Choice('filters3', [64,128]), (3,3), padding='same'),
#         layers.BatchNormalization(),
#         layers.Activation('relu'),
#         layers.Conv2D(hp.Choice('filters4', [64,128]), (3,3), padding='same'),
#         layers.BatchNormalization(),
#         layers.Activation('relu'),
#         layers.MaxPooling2D(),
#         layers.Dropout(hp.Float('dropout2',min_value=0.1, max_value=0.5, step=0.1)),

#         layers.GlobalAveragePooling2D(),

#         layers.Dense(
#             units=hp.Int('units', min_value=32, max_value=256, step=32),
#             kernel_regularizer=tf.keras.regularizers.l2(
#                 hp.Float('l2', 1e-5, 1e-3, sampling='log')
#             )
#         ),
#         layers.BatchNormalization(),
#         layers.Activation('relu'),
#         layers.Dropout(hp.Float('dropout3',min_value=0.1, max_value=0.5, step=0.1)),

#         layers.Dense(8,activation='softmax')
#     ])
#     model.compile(
#         loss='sparse_categorical_crossentropy',
#         optimizer=AdamW(hp.Float('lr',min_value=1e-5, max_value=1e-3, sampling='log')),
#         metrics=['accuracy']
#     )
#     return model

In [13]:
def build_model(hp):
    model = tf.keras.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        data_augmentation,
        # Block 1
        layers.Conv2D(hp.Choice('filters1', [32,64]), (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(hp.Choice('filters2', [32,64]), (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(hp.Choice('filters3', [32,64]), (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(hp.Float('dropout1', 0.1, 0.5, step=0.1)),

        # Block 2
        layers.Conv2D(hp.Choice('filters4', [64,128]), (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(hp.Choice('filters5', [64,128]), (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(hp.Choice('filters6', [64,128]), (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(hp.Float('dropout2', min_value=0.1, max_value=0.5, step=0.1)),

        # Block 3
        layers.Conv2D(hp.Choice('filters7', [128,256]), (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(hp.Choice('filters8', [128,256]), (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(hp.Choice('filters9', [128,256]), (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(hp.Float('dropout3', min_value=0.1, max_value=0.5, step=0.1)),

        layers.GlobalAveragePooling2D(),

        layers.Dense(
            2048,
            kernel_regularizer=tf.keras.regularizers.l2(
                hp.Float('l2_1', 1e-5, 1e-3, sampling='log')
            )
        ),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(hp.Float('dropout4',min_value=0.1, max_value=0.5, step=0.1)),
        
        layers.Dense(1024,
            kernel_regularizer=tf.keras.regularizers.l2(
                hp.Float('l2_2', 1e-5, 1e-3, sampling='log')
            )
        ),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(hp.Float('dropout5',min_value=0.1, max_value=0.5, step=0.1)),
        
        layers.Dense(512,
            kernel_regularizer=tf.keras.regularizers.l2(
                hp.Float('l2_3', 1e-5, 1e-3, sampling='log')
            )
        ),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(hp.Float('dropout6',min_value=0.1, max_value=0.5, step=0.1)),
        
        layers.Dense(8,activation='softmax')
    ])
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=AdamW(hp.Float('lr',min_value=1e-5, max_value=1e-3, sampling='log')),
        metrics=['accuracy']
    )
    return model

In [14]:
'''
Initialize a tuner (here, HyperBand). 
We use objective to specify the objective to select the best models,
and we use max_trials to specify the number of different models to try.
'''
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=20,
    factor=3, # Reduction factor 3x fewer models in the next round
    directory='keras_tuner_results',
    project_name='emotion_recognition'
)

Reloading Tuner from keras_tuner_results\emotion_recognition\tuner0.json


In [ ]:
tuner.search(
    train_dataset,
    epochs=10,
    validation_data=val_dataset
)

In [ ]:
best_hp = tuner.get_best_hyperparameters()[0]
print("Best Hyperparameters: ", best_hp.values)
# train from scratch with best hp
# not get the best model (for correct class weights usage)
best_model = build_model(best_hp)

Best Hyperparameters:  {'filters1': 32, 'dropout1': 0.1, 'filters2': 64, 'dropout2': 0.1, 'units': 160, 'l2': 4.668234933893038e-05, 'dropout3': 0.1, 'lr': 0.001879924731863074, 'tuner/epochs': 20, 'tuner/initial_epoch': 0, 'tuner/bracket': 0, 'tuner/round': 0}


In [16]:
print(best_model)

<Sequential name=sequential_1, built=True>


In [17]:
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
mc = ModelCheckpoint(SAVED_MODEL, monitor='val_accuracy', save_best_only=True)
csv_logger = CSVLogger('cnn_weighted_loss_training_log.csv')
rlr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=5,    
    min_lr=1e-6
)

In [18]:
start = time.time()
history = best_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=[es, mc, rlr, csv_logger]
)
end = time.time()
elapsed_time = end - start

print("Training time: ", time.strftime("%H:%M:%S", time.gmtime(elapsed_time)))

Epoch 1/100
114/403 ━━━━━━━━━━━━━━━━━━━━ 1:48 376ms/step - accuracy: 0.1434 - loss: 2.4282

KeyboardInterrupt: 

In [ ]:
plt.figure(figsize=(12,5))

# Loss
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Accuracy
plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

In [ ]:
model = tf.keras.models.load_model(SAVED_MODEL)

In [ ]:
test_loss, test_acc = model.evaluate(test_dataset)
print(f"Test Accuracy: {test_acc*100:.2f}%")

In [ ]:
def report_test_results():
    print("Evaluating on Test Set...")
    # Evaluate returns [loss, accuracy]
    loss, accuracy = model.evaluate(test_dataset)
    print(f"Test Accuracy: {accuracy*100:.2f}%")

    # Make predictions
    print("Generating predictions...")

    # generator reset to start from the beginning (important for correct label alignment)
    test_dataset.reset()
    predictions = model.predict(test_dataset, verbose=1)

    # Convert predictions to class indexes
    y_pred_indices = np.argmax(predictions, axis=1)

    # Get true labels directly from the generator
    y_true_indices = test_dataset.classes

    # Get the class names (labels)
    class_labels = list(test_dataset.class_indices.keys())

    # Classification Report
    print("\nClassification Report:\n")
    print(classification_report(y_true_indices, y_pred_indices, target_names=class_labels))

    # Confusion Matrix
    cm = confusion_matrix(y_true_indices, y_pred_indices)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix')
    plt.show()

# noisy labels (many classes mix up emotion classification)
report_test_results()

In [ ]:
predictions = model.predict(test_dataset, verbose=1)

# Convert predictions to class indexes
y_pred_indices = np.argmax(predictions, axis=1)

# Get true labels directly from the generator
y_true_indices = test_dataset.classes

# axis 0 and axis 1 depend on the shape of predictions (num_samples, num_classes)
confidence = np.max(predictions, axis=1)
print("Confidence scores for predictions: ", confidence)

wrong_idx = np.where(y_pred_indices != y_true_indices)[0]
print("Total wrong predictions: ", len(wrong_idx))

high_confidence_wrong_idx = wrong_idx[confidence[wrong_idx] > 0.8]
print("High confidence wrong predictions: ", len(high_confidence_wrong_idx))

In [ ]:
def predict_random_samples():
    # Grab a single batch of images
    # We use next() to fetch the first batch from the generator
    images, labels = next(test_dataset)

    # Pick 5 random indices from this batch (batch size is usually 32)
    indices = np.random.choice(len(images), 5, replace=False)

    # Get class names map {0: 'angry', 1: 'happy', ...}
    # class_map = {v: k for k, v in val_dataset.class_indices.items()}

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))

    for i, idx in enumerate(indices):
        img = images[idx]

        # Get True Label
        true_idx = labels[idx]
        true_label = CLASSES[true_idx]

        # Get Prediction
        # Add extra dim because model expects (Batch, Height, Width, Channel)
        pred_prob = model.predict(np.expand_dims(img, axis=0), verbose=0)
        pred_idx = np.argmax(pred_prob)
        pred_label = CLASSES[pred_idx]

        # Display Image
        # Squeeze removes the channel dim (96,96,3) -> (96,96) for plotting
        axes[i].imshow(img.squeeze())
        axes[i].axis('off')

        # Title color: Green if correct, Red if wrong
        color = 'green' if true_label == pred_label else 'red'
        axes[i].set_title(f"True: {true_label}\nPred: {pred_label}", color=color)

    plt.show()

predict_random_samples()